In [1]:
import os
from mp_api.client import MPRester
from pymatgen.analysis.magnetism.analyzer import Ordering 
import pandas as pd

In [2]:
API_KEY = os.getenv("MP_API")
API_KEY

'xURuf206Ym3ILaxe9EDdkrCiTaUBwNUY'

## Data to Fetch

For a permanent magnet ML project, you need materials that are:
- Magnetic (ferromagnetic ideally)
- Contain rare earth or transition metals (Fe, Nd, Co, Sm, etc.)

### Key Properties to Fetch

| Property | Field Name | Why Important |
|----------|-----------|---------------|
| Total magnetization | `total_magnetization` | Strength of magnet |
| Magnetic ordering | `ordering` | FM/AFM/FiM classification |
| Energy above hull | `energy_above_hull` | Material stability |
| Band gap | `band_gap` | Metallic vs insulating |
| Formation energy | `formation_energy_per_atom` | Thermodynamic stability |
| Density | `density` | Physical property |
| Elements present | `elements` | Composition features |
| Crystal system | `crystal_system` | Structure type |
| Volume | `volume` | Unit cell size |
| Number of sites | `nsites` | Complexity |

## Fetch the Data

In [3]:
with MPRester(API_KEY) as mpr:

    results = mpr.materials.summary.search(
        magnetic_ordering=Ordering.FM,
        fields=[
            "material_id",
            "formula_pretty",
            "elements",
            "nelements",
            "nsites",
            "volume",
            "density",
            "energy_above_hull",
            "formation_energy_per_atom",
            "band_gap",
            "total_magnetization",
            "total_magnetization_normalized_vol",
            "ordering",
            "symmetry",          # ✅ replaces crystal_system + spacegroup_number
            "is_magnetic",
            "num_magnetic_sites",
            "types_of_magnetic_species",
        ]
    )

Retrieving SummaryDoc documents:   0%|          | 0/52205 [00:00<?, ?it/s]

In [4]:
f"Fetched {len(results)} materials"

'Fetched 52205 materials'

### Convert to DataFrame

In [ ]:
data = []
for r in results:
    data.append({
        "material_id":              r.material_id,
        "formula":                  r.formula_pretty,
        "elements":                 [str(e) for e in r.elements],
        "nelements":                r.nelements,
        "nsites":                   r.nsites,
        "volume":                   r.volume,
        "density":                  r.density,
        "energy_above_hull":        r.energy_above_hull,
        "formation_energy":         r.formation_energy_per_atom,
        "band_gap":                 r.band_gap,
        "total_magnetization":      r.total_magnetization,
        "magnetization_norm_vol":   r.total_magnetization_normalized_vol,
        "ordering":                 str(r.ordering),
        "crystal_system":           str(r.symmetry.crystal_system) if r.symmetry else None,  # ✅ extracted from symmetry
        "spacegroup_number":        r.symmetry.number if r.symmetry else None,               # ✅ extracted from symmetry
        "spacegroup_symbol":        r.symmetry.symbol if r.symmetry else None,
        "is_magnetic":              r.is_magnetic,
        "num_magnetic_sites":       r.num_magnetic_sites,
        "magnetic_species":         [str(s) for s in r.types_of_magnetic_species] if r.types_of_magnetic_species else [],
    })

df = pd.DataFrame(data)
df.to_csv("./data/magnet_dataset.csv", index=False)
print(df.shape)
df.head()

(52205, 19)


,material_id,formula,elements,nelements,nsites,volume,density,energy_above_hull,formation_energy,band_gap,total_magnetization,magnetization_norm_vol,ordering,crystal_system,spacegroup_number,spacegroup_symbol,is_magnetic,num_magnetic_sites,magnetic_species
0,mp-1183105,Ac3Ce,"[Ac, Ce]",2,4,158.539381,8.600357,0.127472,0.127472,0.0,2.289830,0.014443,FM,Cubic,221,Pm-3m,True,1,[Ce]
1,mp-1183307,Ac3Ce,"[Ac, Ce]",2,4,146.960899,9.277945,0.172509,0.172509,0.0,1.667747,0.011348,FM,Tetragonal,139,I4/mmm,True,1,[Ce]
2,mp-1183673,Ac3Ce,"[Ac, Ce]",2,8,338.954438,8.045301,0.130006,0.130006,0.0,4.157504,0.012266,FM,Hexagonal,194,P6_3/mmc,True,2,[Ce]
3,mp-1183075,Ac3Cr,"[Ac, Cr]",2,4,150.451372,8.090113,0.450683,0.450683,0.0,3.576497,0.023772,FM,Cubic,221,Pm-3m,True,1,[Cr]
4,mp-1183097,Ac3Mo,"[Ac, Mo]",2,4,137.582258,9.377221,0.691670,0.691670,0.0,1.572790,0.011432,FM,Tetragonal,139,I4/mmm,True,1,[Mo]


## Filter for Permanent Magnet Relevant Materials

After fetching, filter to get **ferromagnetic + stable** materials only:


In [6]:
df_pm = df[
    (df["energy_above_hull"] < 0.1) &           # Stable materials
    (df["total_magnetization"] > 0.5)           # Meaningful magnetization
].copy()

print(f"Permanent magnet candidates: {len(df_pm)}")

Permanent magnet candidates: 28472
